# 10 — Pairplot de features (Grid-Finding)

Genera un pairplot de las features extraídas por la pipeline, coloreado por `zone_type`.

Misma estructura que el pairplot del `Tutorial8_DimReduction_OSM.ipynb`:
- Filtra a clases binarias **Commercial vs Residential**
- Submuestrea a 2,000 puntos para que el plot sea legible y rápido
- Imputa NaN con la mediana antes de graficar
- Guarda el output en `outputs/<Borough>/pairplot_features.png`

**Input:** `combined_grid.csv` (output del orchestrator tras mergear 01-06).
**Output:** `pairplot_features.png`

In [ ]:
# ── Papermill parameters ──────────────────────────────
CSV_PATH = "csv/Manhattan/combined_grid.csv"
PLOTS_DIR = "outputs/Manhattan"
RANDOM_STATE = 42
PAIRPLOT_SAMPLE = 2000

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

sns.set_style("darkgrid")
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Cargar el combined_grid

In [ ]:
data_raw = pd.read_csv(CSV_PATH)
print(f"Shape original: {data_raw.shape}")
print(f"Columnas: {data_raw.columns.tolist()}")
data_raw.head(3)

## 2. Filtrar a Commercial vs Residential

In [ ]:
print("Distribución de zone_type:")
print(data_raw['zone_type'].value_counts())

data = data_raw[data_raw['zone_type'].isin(['Commercial', 'Residential'])].copy()
data = data.reset_index(drop=True)
print(f"\nShape tras filtrar: {data.shape}")

## 3. Seleccionar features numéricas

Se excluyen identificadores (`cell_id`), coordenadas (`cell_lat`, `cell_lon`) y la etiqueta (`zone_type`). El resto son las features extraídas por los notebooks 01-06.

In [ ]:
EXCLUDE = {'cell_id', 'cell_lat', 'cell_lon', 'zone_type'}
feature_cols = [c for c in data.columns if c not in EXCLUDE and pd.api.types.is_numeric_dtype(data[c])]
print(f"Features ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  - {f}")

data_num = data[feature_cols].copy()
data_num = data_num.fillna(data_num.median(numeric_only=True))
print(f"\nShape data_num: {data_num.shape}")

## 4. Generar el pairplot

Igual que Tutorial 8: submuestreo a 2,000 puntos, hue = `zone_type`, transparencia 0.4.

In [ ]:
pairplot_n = min(PAIRPLOT_SAMPLE, len(data_num))
pairplot_idx = data_num.sample(n=pairplot_n, random_state=RANDOM_STATE).index
sample_for_pairplot = pd.concat([
    data_num.loc[pairplot_idx],
    data.loc[pairplot_idx, ['zone_type']]
], axis=1)

print(f"Plotting {len(sample_for_pairplot)} puntos x {len(feature_cols)} features...")
g = sns.pairplot(sample_for_pairplot, hue='zone_type', plot_kws={'alpha': 0.4, 's': 12})
g.fig.suptitle('Pairplot — Features Grid-Finding × zone_type', y=1.02)

## 5. Guardar como PNG

In [ ]:
pathlib.Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)
out_path = pathlib.Path(PLOTS_DIR) / 'pairplot_features.png'
g.savefig(out_path, dpi=120, bbox_inches='tight')
print(f"Saved: {out_path}")